## Description

The goal of this notebook is to merge all EEG features, diagnosis, and demog information into 1 CSV file.

# Imports

In [1]:
# Imports
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import FileLink

### Contants

In [2]:
isRio = False # Set to True if running Rio analysis

### Paths

In [3]:
# Directory
root_dir = "/Users/emmanuelle.coutu-nadeau/Library/Mobile Documents/com~apple~CloudDocs/UdeM/MSc Psycho/LABO NED - Personal Drive/Code/GENiAL/"
data_dir = os.path.join(root_dir, 'Data/')

# Original Data CSVs from REDCAP
original_demog_genetics_data = os.path.join(root_dir,'Data/Genetics/Input/Q1K report EEG_NDD_génétique.csv')
original_dia_cogn_data = os.path.join(root_dir, 'Data/Diagnosis + Cogn Tests/Q1K-Dia_cogn.csv')
original_eeg_rs_data = os.path.join(root_dir, 'Data/EEG/Q1K_concatenated_features_RS.csv') # Preprosessed EEG data with HAPPE

# Input Files for CNV prediction
genetics_only_data = os.path.join(root_dir, 'Data/Genetics/Input/CNV-Analysis.csv')
hg38_input_data = os.path.join(root_dir,'Data/Genetics/Input/CNV-Analysis-Hg38.tsv')
hg19_input_data = os.path.join(root_dir,'Data/Genetics/Input/CNV-Analysis-Hg19.tsv')
hg18_input_data = os.path.join(root_dir,'Data/Genetics/Input/CNV-Analysis-Hg18.tsv')

# Calculated CNVs files
cnv_prediction_hg19_data = os.path.join(root_dir, 'Data/Genetics/Input/cnvprediction-hg19-output.csv')
cnv_prediction_hg38_data = os.path.join(root_dir, 'Data/Genetics/Input/cnvprediction-hg38-output.csv')

# Index to Pariticpant Code Map
id_map = os.path.join(root_dir, 'Data/Genetics/Input/sample-id-map.csv')

# Output Files
preprocessed_data_path = os.path.join(root_dir, 'Data/Final/GENIAL-DB-preprocessed.csv')

### Prepare input files for CNV online tool

In [4]:
# CNV data - separated into hg19, hg38, and hg18
# These will be used to input into the CNV prediction tool
genetics_df = pd.read_csv(genetics_only_data)
genetics_df['Human Genome Version'].astype(str)

selected_columns = ['Sample.ID','Sex','CHR','START','STOP','TYPE']
df_38 = genetics_df[genetics_df['Human Genome Version'] == 'Hg38'][selected_columns]
df_19 = genetics_df[genetics_df['Human Genome Version'] == 'Hg19'][selected_columns]
df_18 = genetics_df[genetics_df['Human Genome Version'] == 'Hg18'][selected_columns] # Hg18, ignore

# Save the DataFrame as a TSV file without the index column
df_38.to_csv(hg38_input_data, sep='\t', index=False)
df_19.to_csv(hg19_input_data, sep='\t', index=False)
df_18.to_csv(hg18_input_data, sep='\t', index=False) # Hg18, ignore


### Import files

In [5]:
# ---- Import Data ----
# Original data (CSV)
df = pd.read_csv(original_demog_genetics_data)

# Diagnosis and Cognitive tests Data (CSV)
dia_cogn_df = pd.read_csv(original_dia_cogn_data)

# EEG RS features (CSV)
eeg_rs_features_df = pd.read_csv(original_demog_genetics_data)

# CNV prediction outputs from tool
cnv_hg19_df = pd.read_csv(cnv_prediction_hg19_data)
cnv_hg38_df = pd.read_csv(cnv_prediction_hg38_data)

# Map of # id (used in CNV prediction) and Q1K id
id_map = pd.read_csv(id_map)

# Data manipulations

### Rename columns

In [ ]:
# Keep only first age column
age_cols = [col for col in df.columns if col == "Age in years"]
df = df.rename(columns={age_cols[0]: "Age at EEG (years)"})

# Keep All columns starting with "Diagnosis"
df = df.loc[:, df.columns.str.startswith('Diagnosis')]

# Keep all columns starting with "Inheritance"
df = df.loc[:, df.columns.str.startswith('Inheritance')]

# Rename columns to keep
df = df.rename(columns=
               {'Enter in the box participant\'s EEG code as written here :  [intake_arm1][q1k_relative_idgenerated_1] [intake_arm1][q1k_proband_id_1]': 'ParticipantID',
                'Was EEG attempted?': 'EEG_attempted',
                'EEG site:': 'EEG_site',
                'Birthdate': 'Birthdate',
                'EEG Date': 'EEG_date',
                'Age at EEG (years)': 'EEG_age',
                'Sex at birth:': 'Sex_at_birth',
                'Unknown - Specify:': 'diag_unknown_specify',
                'Other - Specify:': 'diag_other_specify',
                'Medication taken the morning of the EEG': 'medication_at_EEG',
                'Resting state with Rio done?': 'RS_Rio_done',
                'Participant\'s code for resting state with Rio :': 'RS_Rio_code',
                'Resting state done?': 'RS_done',
                'Participant\'s code for resting state :': 'RS_code',
                'Tone Oddball done?': 'TO_done',
                'Participant\'s code for TO': 'TO_code',
                'GO done?': 'GO_done',
                'Participant\'s code for GO:': 'GO_code',
                'VEP done?': 'VEP_done',
                'Participant\'s code for VEP:': 'VEP_code',
                'AEP done?': 'AEP_done',
                'Participant\'s code for AEP :   Choose version A or B': 'AEP_code',
                'Randomization file used (A or B)': 'AEP_randomization_file',
                'NSP done?': 'NSP_done',
                'Participant\'s code for NSP:': 'NSP_code',
                'VS done?': 'VS_done',
                'Participant\'s code for VS:': 'VS_code',
                'MMN Oddball done?': 'MMN_done',
                'Participant\'s code for MMN': 'MMN_code',
                'Result aCGH/ LP-WGS': 'Genetic_test_result',
                'Genetic status of the participant:': 'Genetic_status',
                'Affected chromosome:': 'Affected_chromosome',
                'Full proximal boundary (e.g., 2960000):': 'Proximal_boundary',
                'Full distal boundary (e.g., 3020000):': 'Distal_boundary',
                'Please indicate the Human Genome Version used': 'Genome_version',
                'Single gene testing:': 'Single_gene_testing',
                'Fragile X': 'Fragile_X',
                'Exome / Panel testing:': 'Exome_panel_testing',
                'Diagnosis (choice=Control (no genetic or neurodev disorder))': 'diag_control',
                'Diagnosis (choice=Neurodevelopmental disorder)': 'diag_neurodev',
                'Diagnosis (choice=Genetic carrier)': 'diag_genetic_carrier',
                'Diagnosis (choice=Unknown (under investigation, suspected))': 'diag_unknown',
                'Diagnosis (choice=Other (non neurodevelopmental diagnosis))': 'diag_other',
                'Inheritance (choice=De novo)': 'inheritance_denovo',
                'Inheritance (choice=Mothers inherited)': 'inheritance_mothers_inherited',
                'Inheritance (choice=Fathers inherited)': 'inheritance_fathers_inherited',
                'Inheritance (choice=Unknown)': 'inheritance_unknown',
                'Inheritance (choice=Mosaic)': 'inheritance_mosaic'
                })

# Keep only the renamed columns
df = df[list(column_mapping.values())]


In [ ]:
# ---- New column : Family member type ----
# Function to determine the family_member_type
def categorize_family_member_type(id_value):
    last_part = id_value.split('_')[-1]
    if last_part == 'P':
        return 'Proband'
    elif last_part.startswith('S') and last_part[1:].isdigit():
        return 'Sibling'
    elif last_part.startswith('F') and last_part[1:].isdigit():
        return 'Father'
    elif last_part.startswith('M') and last_part[1:].isdigit():
        return 'Mother'
    elif last_part.startswith('C') and last_part[1:].isdigit():
        return 'Child'
    elif last_part.startswith('O') and last_part[1:].isdigit():
        return 'Other'
    else:
        return pd.NA


df['ParticipantID'] = df['ParticipantID'].astype('str')

# Create new column with family member type
df['family_member_type'] = df['ParticipantID'].apply(categorize_family_member_type)

### Merge CNV to original DF

In [6]:
# Merge participantID to the genetic data
cnv_hg19_df = cnv_hg19_df.merge(id_map, on='ID', how = 'left')
cnv_hg38_df = cnv_hg38_df.merge(id_map, on='ID', how = 'left')

# Merge hg19 and hg38 dataframes
cnv_df = pd.concat([cnv_hg19_df, cnv_hg38_df], axis=0)

# Force ParticipantID to be a string
cnv_df['ParticipantID'] = cnv_df['ParticipantID'].astype(str).str.strip()
df['ParticipantID'] = df['ParticipantID'].astype(str).str.strip()
cnv_df.columns = cnv_df.columns.str.strip()
df.columns = df.columns.str.strip()

In [7]:
# Select genetic columns of interest
selected_columns = ['ParticipantID', 'NVIQ_CIupr', 'ORASD_upr', 'SRS_CIupr', 'PdN_CIupr', 'sum_LOEUF_complete']
cnv_selected = cnv_df[selected_columns]

cnv_selected = cnv_selected.rename(
    columns={
        'NVIQ_CIupr': 'Estimated loss of Non-Verbal Intelligence Quotient',
        'ORASD_upr': 'Estimated odds ratio for autism',
        'SRS_CIupr': 'Estimated gain of raw score of Social Responsiveness Scale',
        'PdN_CIupr': 'Estimated probability of being de novo',
        'sum_LOEUF_complete': 'Sum LOEUF'
    }
)

# Merge
df = df.merge(cnv_selected, on='ParticipantID', how='left')

### Merge diagnosis and cognitive tests

In [8]:
# Strip leading and trailing spaces from all column names
dia_cogn_df.columns = dia_cogn_df.columns.str.strip()

# Create column ParticipantID and make sure no leading or trailing spaces
dia_cogn_df['ParticipantID'] = dia_cogn_df['eeg_participant_code'].astype(str).str.strip()
dia_cogn_df.columns = dia_cogn_df.columns.str.strip()

In [9]:
# Merge the two DataFrames on ParticipantID
merged_df = df.merge(dia_cogn_df, on="ParticipantID", how="left")

In [10]:
# Drop duplicated columns
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]

In [11]:
# Drop the specified columns
columns_to_drop = ['record_id', 'redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance', 'eeg_participant_code']
merged_df = merged_df.drop(columns=columns_to_drop)

In [12]:
# Strip leading and trailing spaces from all string values in the DataFrame
merged_df = merged_df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

/var/folders/7j/mcx19g313_vgs3_tv_rpqmrw0000gn/T/ipykernel_39553/4041133145.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  merged_df = merged_df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


### Merge RS EEG features

In [13]:
# Convert CSV to dataframe
rs_eeg_features_df = pd.read_csv(original_eeg_rs_data)

# Create 2 distinct df for RS Rio vs pure RS
rs_df = rs_eeg_features_df[rs_eeg_features_df['ID'].str.contains('_RS_')].copy()
rsrio_df = rs_eeg_features_df[rs_eeg_features_df['ID'].str.contains('_RSRIO_')].copy()

# Cleanup participant ID
rs_df['ParticipantID'] = rs_df['ID'].str.replace(r'_RS_.*', '', regex=True)
rsrio_df['ParticipantID'] = rsrio_df['ID'].str.replace(r'_RSRIO_.*', '', regex=True)

# Identify EEG features columns
rs_df = rs_df.rename(columns={col: f"EEG_{col}" for col in rs_df.columns if col not in ['ID', 'ParticipantID']})
rsrio_df = rsrio_df.rename(columns={col: f"EEG_{col}" for col in rsrio_df.columns if col not in ['ID', 'ParticipantID']})


In [14]:
# Merge the RS dataframe on ParticipantID
if isRio: rs_data = rsrio_df
else: rs_data = rs_df

merged_df = merged_df.merge(rs_data, on="ParticipantID", how="left")

# Identify neurodev diagnosis or not

In [15]:
# List of columns to check
columns_to_check = ["diag_asd", "diag_intel", "diag_adhd", "diag_fas", "diag_learn", "diag_comm", "diag_motor"]

# Create the neurodevDiag column
final_df = merged_df.copy()

# Convert columns to numeric if they are not already
for col in columns_to_check:
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce').fillna(0).astype(int)

final_df['diag_neurodev'] = final_df[columns_to_check].apply(lambda row: 1 if (row == 2).any() else 0, axis=1)

## Download DB as CSV

In [16]:
final_df.to_csv(preprocessed_data_path, index=False)
FileLink(preprocessed_data_path)

/Users/emmanuelle.coutu-nadeau/Library/Mobile Documents/com~apple~CloudDocs/UdeM/MSc Psycho/LABO NED - Personal Drive/Code/GENiAL/Data/Final/GENIAL-DB-preprocessed.csv

# Diving into the data...

## Create a subset for probands only

Remove rows without genetic testing and identify those with normal VS abnormal (or VUS = variant of uncertain significance) genetic status

In [22]:
# Subset with only final_df['family_member_type'] == 'Proband'
proband_df = final_df[final_df['family_member_type'] == 'Proband']
proband_df = proband_df.drop(columns=['family_member_type'])

Count: 62 probands

In [23]:
# Remove rows where Genetic Status is missing
proband_df = proband_df[proband_df['Genetic Status'].notna()]

Count: 38 probands with genetic testing

In [33]:
proband_df['Genetic Status'].astype(str)
proband_df['diag_genetic'] = proband_df['Genetic Status'].apply(lambda x: 0 if x == 'Normal' else 1)
count_genetic_diag = (proband_df['diag_genetic'] == 1).sum()

print(f"Number of probands with genetic abnormality: {count_genetic_diag}")

Number of probands with genetic abnormality: 26


## Other family members with diagnostic

In [35]:
# Subset of other family members
other_fam_df = final_df[final_df['family_member_type'] != 'Proband']

# Neurodev count
count_neurodev_diag = (other_fam_df['diag_neurodev'] == 1).sum()
print(f"Number of other family members with neurodevelopmental diagnosis: {count_neurodev_diag}")

# Remove rows where Genetic Status is missing
other_fam_df = other_fam_df[other_fam_df['Genetic Status'].notna()]

# Genetic abnormality count
other_fam_df['diag_genetic'] = other_fam_df['Genetic Status'].apply(lambda x: 0 if x == 'Normal' else 1)
count_genetic_diag = (other_fam_df['diag_genetic'] == 1).sum()

print(f"Number of other family members with genetic abnormality: {count_genetic_diag}")

Number of other family members with neurodevelopmental diagnosis: 5
Number of other family members with genetic abnormality: 7
